# KV-Cache Compression Results Analysis

Visualize and compare baseline vs compressed models.

**Run experiments first**:
```bash
# Baseline
python scripts/run_baseline.py --output results/baseline.json

# 2x compression
python scripts/run_compression.py --compression simple_2x --output results/compression_2x.json

# Adaptive compression  
python scripts/run_compression.py --compression adaptive --output results/compression_adaptive.json
```

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## Load Results

In [ ]:
def load_results(filepath):
    """Load experiment results from JSON."""
    with open(filepath, 'r') as f:
        return json.load(f)

# Load all result files
results_dir = Path("/workspace/padic-transformers/results")

baseline = load_results(results_dir / "baseline.json")
compression_2x = load_results(results_dir / "compression_2x.json")
compression_adaptive = load_results(results_dir / "compression_adaptive.json")

print("✓ Loaded results for:")
print(f"  - Baseline: {baseline['model']}")
print(f"  - 2x Compression: {compression_2x['compression']}")
print(f"  - Adaptive: {compression_adaptive['compression']}")

## Comparison Table

In [ ]:
# Create comparison dataframe
data = []

for ctx_len in baseline['results'].keys():
    ctx = int(ctx_len)
    
    # Baseline
    data.append({
        'Context Length': ctx,
        'Method': 'Baseline (FP16)',
        'Perplexity': baseline['results'][ctx_len]['perplexity'],
        'Memory (MB)': baseline['results'][ctx_len]['peak_memory_mb'],
        'Compression': '1.0x',
    })
    
    # 2x compression
    if ctx_len in compression_2x['results']:
        res = compression_2x['results'][ctx_len]
        data.append({
            'Context Length': ctx,
            'Method': '2x (8-bit)',
            'Perplexity': res['perplexity'],
            'Memory (MB)': res['peak_memory_mb'],
            'Compression': f"{res['compression_ratio']:.1f}x",
        })
    
    # Adaptive compression
    if ctx_len in compression_adaptive['results']:
        res = compression_adaptive['results'][ctx_len]
        data.append({
            'Context Length': ctx,
            'Method': 'Adaptive (16/8/4)',
            'Perplexity': res['perplexity'],
            'Memory (MB)': res['peak_memory_mb'],
            'Compression': f"{res['compression_ratio']:.1f}x",
        })

df = pd.DataFrame(data)
df

## Plot: Perplexity Comparison

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot perplexity for each method
for method in df['Method'].unique():
    subset = df[df['Method'] == method]
    ax.plot(subset['Context Length'], subset['Perplexity'], 
            marker='o', label=method, linewidth=2, markersize=8)

ax.set_xlabel('Context Length (tokens)', fontsize=13)
ax.set_ylabel('Perplexity', fontsize=13)
ax.set_title('Perplexity vs Context Length', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'perplexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to results/perplexity_comparison.png")

## Plot: Memory Usage

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot memory for each method
for method in df['Method'].unique():
    subset = df[df['Method'] == method]
    ax.plot(subset['Context Length'], subset['Memory (MB)'], 
            marker='s', label=method, linewidth=2, markersize=8)

ax.set_xlabel('Context Length (tokens)', fontsize=13)
ax.set_ylabel('Peak Memory (MB)', fontsize=13)
ax.set_title('Memory Usage vs Context Length', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to results/memory_comparison.png")

## Plot: Accuracy vs Memory Tradeoff

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Scatter plot: memory vs perplexity
colors = {'Baseline (FP16)': 'blue', '2x (8-bit)': 'orange', 'Adaptive (16/8/4)': 'green'}

for method in df['Method'].unique():
    subset = df[df['Method'] == method]
    ax.scatter(subset['Memory (MB)'], subset['Perplexity'], 
               s=200, alpha=0.6, label=method, color=colors[method])
    
    # Annotate with context length
    for _, row in subset.iterrows():
        ax.annotate(f"{row['Context Length']//1000}k", 
                   (row['Memory (MB)'], row['Perplexity']),
                   fontsize=9, ha='center')

ax.set_xlabel('Peak Memory (MB)', fontsize=13)
ax.set_ylabel('Perplexity', fontsize=13)
ax.set_title('Accuracy-Memory Tradeoff', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Add "better" arrow annotations
ax.annotate('Better →', xy=(0.02, 0.98), xycoords='axes fraction', 
           fontsize=10, ha='left', va='top', color='green', weight='bold')

plt.tight_layout()
plt.savefig(results_dir / 'accuracy_memory_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to results/accuracy_memory_tradeoff.png")

## Summary Statistics

In [ ]:
print("="*70)
print("SUMMARY: Compression vs Baseline")
print("="*70)

# For largest context length
max_ctx = max([int(k) for k in baseline['results'].keys()])
max_ctx_str = str(max_ctx)

baseline_ppl = baseline['results'][max_ctx_str]['perplexity']
baseline_mem = baseline['results'][max_ctx_str]['peak_memory_mb']

print(f"\nContext length: {max_ctx} tokens\n")

for name, results in [('2x Compression', compression_2x), 
                       ('Adaptive', compression_adaptive)]:
    if max_ctx_str in results['results']:
        res = results['results'][max_ctx_str]
        ppl = res['perplexity']
        mem = res['peak_memory_mb']
        comp_ratio = res.get('compression_ratio', 1.0)
        
        ppl_delta = ((ppl - baseline_ppl) / baseline_ppl) * 100
        mem_savings = ((baseline_mem - mem) / baseline_mem) * 100
        
        print(f"{name}:")
        print(f"  Perplexity: {ppl:.4f} ({ppl_delta:+.2f}% vs baseline)")
        print(f"  Memory: {mem:.2f} MB ({mem_savings:.1f}% savings)")
        print(f"  Compression ratio: {comp_ratio:.2f}x")
        print()

print("="*70)

## Export for Paper

Generate LaTeX table for paper.

In [ ]:
# Create paper-ready table
paper_data = []

for ctx_len in sorted([int(k) for k in baseline['results'].keys()]):
    ctx_str = str(ctx_len)
    
    row = {'Context': f"{ctx_len//1000}k"}
    
    # Baseline
    row['Baseline PPL'] = f"{baseline['results'][ctx_str]['perplexity']:.2f}"
    row['Baseline Mem'] = f"{baseline['results'][ctx_str]['peak_memory_mb']:.1f}"
    
    # 2x
    if ctx_str in compression_2x['results']:
        res = compression_2x['results'][ctx_str]
        row['2x PPL'] = f"{res['perplexity']:.2f}"
        row['2x Mem'] = f"{res['peak_memory_mb']:.1f}"
        row['2x Comp'] = f"{res['compression_ratio']:.1f}x"
    
    # Adaptive
    if ctx_str in compression_adaptive['results']:
        res = compression_adaptive['results'][ctx_str]
        row['Adaptive PPL'] = f"{res['perplexity']:.2f}"
        row['Adaptive Mem'] = f"{res['peak_memory_mb']:.1f}"
        row['Adaptive Comp'] = f"{res['compression_ratio']:.1f}x"
    
    paper_data.append(row)

paper_df = pd.DataFrame(paper_data)
print("\nPaper Table (CSV):")
print(paper_df.to_csv(index=False))

# Save
paper_df.to_csv(results_dir / 'paper_table.csv', index=False)
print("\n✓ Saved to results/paper_table.csv")

## Conclusions

Write your observations here after running experiments.